# 졸업프로젝트 — 전체 실험 노트북 (통합본)

> **이 파일은 `scripts/merge_notebooks.py` 가 생성한다. 직접 고치지 말 것.**
> 원본은 같은 폴더의 `0_analysis` / `1_wsd_grid` / `2_vit` / `3_muon` 이다.
> 실제 실행은 원본 4개를 따로 쓰는 편이 낫다 (Colab 세션이 약 80분마다 끊긴다).

## 주제

SAM 계열 평탄화(flatness)가 **언제 실제로 일반화에 기여하는가**.

CIFAR-10 / ResNet-18 에서 WSD 학습률 스케줄의 decay 구간에만 SAM 을 켜고,
같은 Stage 1 체크포인트에서 ρ ∈ {0, 0.02, 0.05, 0.1} 로 분기시켜 비교한다.

## 지금까지의 결과 (완주 arm 14개 / seed 42~45)

| ρ | n | 곡률 감소 | 정확도 증가 | Δacc |
|---|---|---|---|---|
| 0.02 | 4 | **4/4** | 1/4 | +0.02 %p |
| 0.05 | 3 | **3/3** | 1/3 | +0.09 %p |
| 0.10 | 3 | **3/3** | **3/3** | +0.26 %p |

- **곡률 감소는 10/10 arm 에서 예외 없다.** 평균이 ρ 에 완벽히 단조 (0.088 → 0.020).
- 정확도는 ρ=0.10 에서 3/3 개선되나 p=0.125 로 아직 유의하지 않다. 5시드 필요.

## 구조

각 섹션은 **자체완결형**이다. 정의가 겹치지만 각 섹션이 실행 직전에
필요한 것을 전부 재정의하므로, 위에서 아래로 실행하면 각자 자기 정의로 돈다.

## 이 코드가 아카이브 노트북과 다른 점

`GraduationProject/1_baseline`~`4_wsd` 의 옛 노트북에는 결함 세 가지가 있고
여기서 전부 고쳤다 (자세한 내용은 `docs/PROGRESS.md` §6).

1. **Stage 2 에 시드가 없었다** — 두 arm 이 서로 다른 데이터 순서를 봤다.
   "차이는 오직 SAM 유무"가 성립하지 않았다.
2. **probe 가 전역 RNG 에 의존** — 세션마다 다른 이미지로 sharpness 를 쟀다.
3. **측정식의 1차항이 grad norm** — `L(w+ε)−L(w) ≈ ρ‖g‖ + ½ρ²(gᵀHg)/‖g‖²`.
   대칭 차분 `½(L(w+ε)+L(w−ε))−L(w)` 로 1차항을 상쇄해 함께 기록한다.

---

# ==============================================================================
# 섹션 1 — Phase 1 — WSD × SAM 메인 그리드
# ==============================================================================

GPU 필요. 프로젝트의 핵심 실험. 시드당 약 65분.

원본: `1_wsd_grid.ipynb`  ·  셀 16개

# Phase 1 — WSD × SAM 메인 그리드

프로젝트의 핵심 실험. `sam_off` / ρ=0.02 / 0.05 / 0.1 을 **같은 Stage 1 체크포인트에서
분기**시켜 시드별로 반복한다.

| | |
|---|---|
| 소요 | 시드당 약 65분 (Stage 1 15분 + arm 4개) |
| 5시드 | 약 5.5시간 |
| 판정 | Gate A 는 첫 arm(≈23분)에 가능, Gate B/C 는 완주 후 |

**중단돼도 괜찮다.** arm 이 끝날 때마다 JSON 이 저장되고, 다시 실행하면 건너뛴다.

이 노트북은 **자체완결형**이다. 외부 `.py` 없이 혼자 돈다.
노트북판의 결함 세 가지(Stage 2 시드 미고정 · probe 의 전역 RNG 의존 ·
sharpness 측정식의 1차항)는 아래 정의 셀에서 모두 교정됐다.
자세한 내용은 `docs/PROGRESS.md` §4.2, §6.

> ⚠️ 세션을 끄기 전에 **마지막 셀에서 결과를 반드시 회수**할 것.

In [ ]:
# ══ 1. 셋업 ═══════════════════════════════════════════════════════════════
import os, sys, math, json, time, glob, random, tarfile, urllib.request
from datetime import datetime
from collections import defaultdict

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ── 결과 저장 위치 ────────────────────────────────────────────────────────
# /content 는 세션이 끊기면 사라진다. 어느 쪽으로 돌리느냐에 따라 다르다.
#
#   CLI (colab exec)  : 기본값 그대로. 세션을 살려두고 끝나면 colab download 로 회수.
#                       drive.mount 는 대화형이라 헤드리스에서 멈출 수 있으므로 쓰지 않는다.
#   Colab 웹에서 5시간 : USE_DRIVE = True 로 두면 중단돼도 이어서 갈 수 있다.
USE_DRIVE = False

OUT_DIR = '/content/out'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/wsd_sam/out'
    except Exception as e:
        print('Drive 마운트 실패 → /content/out 사용:', e)

DATA_DIR = os.environ.get('DATA_DIR', '/content/data')
OUT_DIR  = os.environ.get('OUT_DIR', OUT_DIR)
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device,
      f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else '')
print('torch  :', torch.__version__)
print('출력   :', OUT_DIR)
if device.type != 'cuda':
    print('\n⚠️ GPU 가 아니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿀 것.')

In [ ]:
# ══ 2. 재현성 ═════════════════════════════════════════════════════════════
# 노트북판의 가장 큰 결함이 "Stage 2 에 시드가 없다"는 것이었다.
# sam_off 셀이 RNG 를 소모한 뒤 sam_on 셀이 시작돼, 두 arm 이 서로 다른
# 데이터 순서를 봤다. 아래 세 함수로 그 경로를 원천 차단한다.
def seed_everything(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

def worker_init(worker_id):
    random.seed(torch.initial_seed() % 2**32 + worker_id)

print('시드 유틸 준비 완료')

In [ ]:
# ══ 3. 데이터 ═════════════════════════════════════════════════════════════
CIFAR_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
C10_MEAN, C10_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)

def prepare_data(data_dir=DATA_DIR):
    root = os.path.join(data_dir, 'cifar10')
    if not os.path.exists(root):
        tgz = os.path.join(data_dir, 'cifar10.tgz')
        if not os.path.exists(tgz):
            print('CIFAR-10 다운로드 중...', flush=True)
            urllib.request.urlretrieve(CIFAR_URL, tgz)
        print('압축 해제 중...', flush=True)
        with tarfile.open(tgz, 'r:gz') as tar:
            tar.extractall(path=data_dir, filter='data')
    return root

def build_loaders(seed, batch_size=128, num_workers=2):
    root = prepare_data()
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    tf_eval = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    train = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_train)
    test  = datasets.ImageFolder(os.path.join(root, 'test'),  transform=tf_eval)
    # probe 전용: 같은 train 이미지지만 augmentation 없이 결정론적으로 읽는다
    probe_src = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_eval)
    tl = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=True, generator=loader_generator(seed),
                    worker_init_fn=worker_init)
    vl = DataLoader(test, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tl, vl, probe_src

def make_probe(probe_src, n=384, batch_size=128, probe_seed=1234):
    """sharpness 측정용 고정 배치.
    인덱스는 별도 Generator 로 뽑고 이미지는 augmentation 이 없으므로,
    세션이 달라도 항상 같은 텐서가 나온다."""
    g = torch.Generator().manual_seed(probe_seed)
    idx = torch.randperm(len(probe_src), generator=g)[:n].tolist()
    loader = DataLoader(Subset(probe_src, idx), batch_size=batch_size, shuffle=False)
    return [(x.to(device), y.to(device)) for x, y in loader]

print('데이터 함수 준비 완료')

In [ ]:
# ══ 4. 옵티마이저 + SAM ═══════════════════════════════════════════════════
class Lion(optim.Optimizer):
    """update = sign(b1*m + (1-b1)*g),  m = b2*m + (1-b2)*g"""
    def __init__(self, params, lr=2e-4, betas=(0.9, 0.99), weight_decay=0.5):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            b1, b2 = group['betas']
            for p in group['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'exp_avg' not in st: st['exp_avg'] = torch.zeros_like(p)
                m, g = st['exp_avg'], p.grad
                if wd: p.mul_(1 - lr * wd)
                p.add_(m.mul(b1).add_(g, alpha=1-b1).sign_(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1-b2)


class SAMAscent:
    """SAM 을 '옵티마이저'가 아니라 '그래디언트 변환기'로 분리한 것.
    perturbation 만 담당하고 step 은 외부 옵티마이저가 하므로,
    옵티마이저를 교체하지 않고 SAM 을 켜고 끌 수 있다 → 모멘텀이 연속된다."""
    def __init__(self, params, rho=0.0):
        self.params = [p for p in params if p.requires_grad]
        self.rho = rho; self._e_w = {}

    def set_rho(self, rho): self.rho = rho

    @torch.no_grad()
    def grad_norm(self):
        gs = [p.grad.norm(p=2) for p in self.params if p.grad is not None]
        return torch.norm(torch.stack(gs), p=2) if gs else torch.tensor(0.0)

    @torch.no_grad()
    def ascent(self):
        if self.rho <= 0: return False
        gn = self.grad_norm()
        if gn.item() < 1e-12: return False
        scale = self.rho / (gn + 1e-12)
        self._e_w.clear()
        for p in self.params:
            if p.grad is None: continue
            e = p.grad * scale.to(p.device); p.add_(e); self._e_w[p] = e
        return True

    @torch.no_grad()
    def restore(self):
        for p, e in self._e_w.items(): p.sub_(e)
        self._e_w.clear()

def build_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)

print('Lion / SAMAscent / ResNet-18 준비 완료')

In [ ]:
# ══ 5. WSD 스케줄 + sharpness 측정 ════════════════════════════════════════
def wsd_lr(epoch, total_epochs, peak_lr, warmup_epochs, decay_epochs):
    """warmup(선형 상승) → stable(유지) → decay(선형 하강)"""
    stable_end = total_epochs - decay_epochs
    if epoch < warmup_epochs:
        return peak_lr * (epoch + 1) / warmup_epochs
    if epoch < stable_end:
        return peak_lr
    prog = min(1.0, max(0.0, (epoch - stable_end + 1) / decay_epochs))
    return peak_lr * (1.0 - prog)


@torch.no_grad()
def _perturb(e_ws, sign):
    for p, e in e_ws: p.add_(e, alpha=sign)


def measure_sharpness(model, criterion, probe, rho=0.05):
    """세 값을 함께 반환한다.

    one_sided : L(w+e) − L(w)
        테일러 전개하면 ≈ rho*||g|| + (1/2)rho^2 * (g'Hg)/||g||^2 이다.
        즉 1차항이 그래디언트 노름이라, 이 값은 곡률이 아니라 ||g|| 를
        상당 부분 재고 있다. 과거 결과와 비교하려고 유지한다.

    symmetric : (1/2)(L(w+e) + L(w−e)) − L(w)
        1차항이 상쇄되어 곡률 항만 남는다. 이쪽이 우리가 재려던 것.

    grad_norm : 위 해석을 사후 검증하기 위한 원자료 (Gate A).
    """
    was_training = model.training
    model.eval()
    one, sym, gns = [], [], []
    for x, y in probe:
        with torch.no_grad():
            l0 = criterion(model(x), y).item()
        model.zero_grad(set_to_none=True)
        criterion(model(x), y).backward()
        gs = [p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]
        gn = torch.norm(torch.stack(gs), p=2); gns.append(gn.item())
        scale = rho / (gn + 1e-12)
        e_ws = [(p, p.grad * scale) for p in model.parameters() if p.grad is not None]
        with torch.no_grad():
            _perturb(e_ws, +1.0);  lp = criterion(model(x), y).item()
            _perturb(e_ws, -2.0);  lm = criterion(model(x), y).item()
            _perturb(e_ws, +1.0)                     # 원위치 복원
        model.zero_grad(set_to_none=True)
        one.append(lp - l0); sym.append(0.5 * (lp + lm) - l0)
    if was_training: model.train()
    n = len(one)
    return sum(one)/n, sum(sym)/n, sum(gns)/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        total += y.size(0); correct += model(x).max(1)[1].eq(y).sum().item()
    return 100.0 * correct / total

print('스케줄 / sharpness / 평가 함수 준비 완료')

In [ ]:
# ══ 6. 실시간 그래프 ══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 기본값은 꺼둔다. 켜면 clear_output 으로 화면을 새로 그리기 때문에
# 에폭 로그가 쌓이지 않고 매번 최신 한 줄만 남는다.
# 로그를 한 줄씩 쌓아 보려면 그대로 두고, 그래프가 필요하면 'on': True 로.
LIVE = {'on': False, 'every': 1}

def live_plot(tag, history, sharp):
    if not LIVE['on'] or len(history['epoch']) % LIVE['every']: return
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history['epoch'], history['train_acc'], lw=1.2, label='train')
    ax[0].plot(history['epoch'], history['test_acc'],  lw=1.6, label='test')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
    ax[0].set_title(f'{tag} — test {history["test_acc"][-1]:.2f}%')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if sharp['epoch']:
        ax[1].plot(sharp['epoch'], sharp['one_sided'], 'o-', ms=3,
                   color='tab:orange', label='one-sided')
        ax[1].plot(sharp['epoch'], sharp['symmetric'], 's-', ms=3,
                   color='tab:purple', label='symmetric (curvature)')
        ax[1].set_xlabel('epoch'); ax[1].set_ylabel('sharpness')
        ax[1].set_title('sharpness (lower = flatter)')
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(history['epoch'], history['grad_norm'], color='tab:brown', lw=1.2)
    ax[2].set_xlabel('epoch'); ax[2].set_ylabel('grad norm', color='tab:brown')
    ax[2].grid(alpha=.3); ax[2].set_title('grad norm & rho')
    a2 = ax[2].twinx(); a2.plot(history['epoch'], history['rho'],
                                color='tab:green', lw=1.4)
    a2.set_ylabel('rho', color='tab:green')
    plt.tight_layout(); plt.show()
    print(f'[{tag}] epoch {history["epoch"][-1]} | '
          f'loss {history["train_loss"][-1]:.4f} | '
          f'test {history["test_acc"][-1]:.2f}% | '
          f'{sum(history["time"])/60:.1f}분 경과', flush=True)

print('그래프 훅 준비 완료 — 기본 꺼짐. 켜려면 LIVE["on"] = True')

In [ ]:
# ══ 7. 학습 루프 ══════════════════════════════════════════════════════════
# Stage 1 과 Stage 2 가 이 함수 하나를 공유한다.
# 노트북판처럼 셀을 복사해 플래그만 바꾸는 구조가 아니므로,
# 두 arm 의 코드 경로가 완전히 동일함이 구조적으로 보장된다.
def new_history():
    keys = ['epoch','phase','lr','rho','train_loss','train_acc','test_acc',
            'grad_norm','ascent_gap','time']
    return ({k: [] for k in keys},
            {'epoch': [], 'one_sided': [], 'symmetric': [], 'probe_grad_norm': []})


def run_epochs(model, optimizer, sam, criterion, train_loader, test_loader,
               probe, epoch_range, cfg, history, sharp, rho_fn, tag):
    stable_end = cfg['total_epochs'] - cfg['decay_epochs']
    last = epoch_range[-1]
    for epoch in epoch_range:
        lr_now = wsd_lr(epoch, cfg['total_epochs'], cfg['peak_lr'],
                        cfg['warmup_epochs'], cfg['decay_epochs'])
        for g in optimizer.param_groups: g['lr'] = lr_now
        rho_now = rho_fn(epoch) if rho_fn else 0.0
        sam.set_rho(rho_now)

        model.train()
        run_loss = run_gap = run_gn = 0.0; correct = total = nb = 0
        t0 = time.time()
        for x, y in train_loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            out = model(x); loss = criterion(out, y); loss.backward()
            total += y.size(0); correct += out.max(1)[1].eq(y).sum().item()
            run_gn += sam.grad_norm().item()

            gap = 0.0
            if sam.ascent():                       # w -> w+e
                optimizer.zero_grad(set_to_none=True)
                loss_adv = criterion(model(x), y); loss_adv.backward()
                gap = loss_adv.item() - loss.item()
                sam.restore()                      # w+e -> w
            if cfg['grad_clip']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            optimizer.step()
            run_loss += loss.item(); run_gap += gap; nb += 1

        dt = time.time() - t0
        tr_loss, tr_acc = run_loss/nb, 100.0*correct/total
        te_acc = evaluate(model, test_loader)
        phase = ('warmup' if epoch < cfg['warmup_epochs']
                 else 'stable' if epoch < stable_end else 'decay')
        for k, v in [('epoch',epoch+1),('phase',phase),('lr',lr_now),('rho',rho_now),
                     ('train_loss',tr_loss),('train_acc',tr_acc),('test_acc',te_acc),
                     ('grad_norm',run_gn/nb),('ascent_gap',run_gap/nb),('time',dt)]:
            history[k].append(v)

        extra = ''
        if (epoch+1) % cfg['sharpness_interval'] == 0 or epoch == last:
            o, s, g = measure_sharpness(model, criterion, probe, cfg['sharpness_rho'])
            sharp['epoch'].append(epoch+1); sharp['one_sided'].append(o)
            sharp['symmetric'].append(s);   sharp['probe_grad_norm'].append(g)
            extra = f' | Sharp1 {o:.4f} | SharpSym {s:.6f} | pg {g:.4f}'
        print(f'[{tag}|{phase:6s}] Ep [{epoch+1}/{cfg["total_epochs"]}] | {dt:.1f}s | '
              f'Loss {tr_loss:.4f} | Train {tr_acc:.2f}% | Test {te_acc:.2f}% | '
              f'lr {lr_now:.5f} | rho {rho_now:.4f} | gn {run_gn/nb:.3f}{extra}', flush=True)
        try:
            live_plot(tag, history, sharp)
        except Exception as e:
            print('  (그래프 오류 무시:', e, ')', flush=True)

print('학습 루프 준비 완료')

In [ ]:
# ══ 8. Stage 1 / Stage 2 ══════════════════════════════════════════════════
def stage1(seed, cfg, force=False):
    """warmup+stable. 시드당 한 번만 학습하고 체크포인트로 캐시한다.
    이 체크포인트를 모든 arm 이 공유하므로 rho 스윕이 거의 공짜로 얹힌다."""
    path = os.path.join(OUT_DIR, f'stage1_seed{seed}.pt')
    if os.path.exists(path) and not force:
        print(f'Stage 1 캐시 재사용: {path}', flush=True); return path

    seed_everything(seed)
    tl, vl, ps = build_loaders(seed, cfg['batch_size'])
    probe = make_probe(ps)
    model = build_model()
    opt = Lion(model.parameters(), lr=cfg['peak_lr'], weight_decay=cfg['weight_decay'])
    sam = SAMAscent(model.parameters()); crit = nn.CrossEntropyLoss()
    hist, sharp = new_history()

    print(f'\n=== Stage 1 (seed {seed}) — {cfg["stable_end"]} epochs ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, probe,
               range(cfg['stable_end']), cfg, hist, sharp, None, f'S1-s{seed}')
    torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(),
                'cfg': cfg, 'seed': seed, 'history': hist,
                'sharpness_history': sharp}, path)
    print(f'\nStage 1 저장: {path}', flush=True)
    return path


def stage2(seed, rho, cfg, ckpt_path):
    """decay 구간. Stage 1 체크포인트에서 분기한다."""
    tag = f'sam{rho:g}' if rho > 0 else 'samoff'
    arm = f'seed{seed}_{tag}_d{cfg["decay_epochs"]}'
    out = os.path.join(OUT_DIR, f'arm_{arm}.json')
    if os.path.exists(out):
        print(f'skip {arm} (이미 있음)', flush=True)
        return json.load(open(out, encoding='utf-8'))

    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    stable_end = ck['cfg']['stable_end']

    # ★ 여기가 노트북판에 없던 부분.
    #   시드를 다시 고정해 모든 arm 이 동일한 데이터 순서를 보게 한다.
    seed_everything(seed + 10_000)
    tl, vl, ps = build_loaders(seed + 10_000, cfg['batch_size'])
    probe = make_probe(ps)

    model = build_model(); model.load_state_dict(ck['model'])
    opt = Lion(model.parameters(), lr=cfg['peak_lr'], weight_decay=cfg['weight_decay'])
    opt.load_state_dict(ck['optimizer'])          # 모멘텀까지 복원
    sam = SAMAscent(model.parameters()); crit = nn.CrossEntropyLoss()
    hist  = {k: list(v) for k, v in ck['history'].items()}
    sharp = {k: list(v) for k, v in ck['sharpness_history'].items()}

    def rho_fn(ep):
        if rho <= 0: return 0.0
        prog = (ep - stable_end + 1) / max(1, cfg['rho_ramp_epochs'])
        return rho * min(1.0, max(0.0, prog))

    a0, o0, s0 = hist['test_acc'][-1], sharp['one_sided'][-1], sharp['symmetric'][-1]
    print(f'\n=== Stage 2 [{arm}] — 출발점 ep{stable_end}: '
          f'Test {a0:.2f}% | SharpSym {s0:.6f} ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, probe,
               range(stable_end, cfg['total_epochs']), cfg, hist, sharp,
               rho_fn, f'S2-{arm}')

    best = max(hist['test_acc'])
    rec = {'arm': arm, 'seed': seed, 'rho_max': rho, 'cfg': cfg,
           'decay_epochs': cfg['decay_epochs'], 'stable_end': stable_end,
           'results': {
               'decay_start_test_acc': a0, 'decay_start_one_sided': o0,
               'decay_start_symmetric': s0,
               'final_train_acc': hist['train_acc'][-1],
               'final_test_acc': hist['test_acc'][-1], 'best_test_acc': best,
               'best_epoch': hist['epoch'][hist['test_acc'].index(best)],
               'final_gap': hist['train_acc'][-1] - hist['test_acc'][-1],
               'final_one_sided': sharp['one_sided'][-1],
               'final_symmetric': sharp['symmetric'][-1],
               'final_probe_grad_norm': sharp['probe_grad_norm'][-1],
               'decay_time_min': sum(hist['time'][stable_end:]) / 60},
           'history': hist, 'sharpness_history': sharp,
           'timestamp': datetime.now().isoformat(timespec='seconds')}
    json.dump(rec, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    r = rec['results']
    print(f'\n=== [{arm}] 완료 ===', flush=True)
    print(f'  Test     {a0:.2f}% -> {r["final_test_acc"]:.2f}% '
          f'({r["final_test_acc"]-a0:+.2f}%p)', flush=True)
    print(f'  SharpSym {s0:.6f} -> {r["final_symmetric"]:.6f}', flush=True)
    print(f'  {r["decay_time_min"]:.1f}분 | 저장: {out}', flush=True)
    return rec

print('Stage 함수 준비 완료')

In [ ]:
# ══ 게이트 판정 함수 ══════════════════════════════════════════════════════
# 기준은 실행 전에 확정됐다 (docs/PROGRESS.md §10 사전등록).
# 결과를 보고 해석을 만드는 게 아니라, 미리 적어 둔 기준에 데이터를 넣는다.
def mean(v): return sum(v)/len(v) if v else float('nan')
def sd(v):
    if len(v) < 2: return float('nan')
    m = mean(v); return (sum((x-m)**2 for x in v)/(len(v)-1))**.5
def pearson(x, y):
    if len(x) < 3: return float('nan')
    mx, my = mean(x), mean(y)
    sx = sum((a-mx)**2 for a in x)**.5; sy = sum((b-my)**2 for b in y)**.5
    if sx == 0 or sy == 0: return float('nan')
    return sum((a-mx)*(b-my) for a,b in zip(x,y))/(sx*sy)
def sign_test_p(k, n):
    return sum(math.comb(n,i) for i in range(k, n+1))/2**n if n else float('nan')

def load_arms(d=None):
    d = d or OUT_DIR
    return [json.load(open(p, encoding='utf-8'))
            for p in sorted(glob.glob(os.path.join(d, 'arm_*.json')))]

def gate_a(arms):
    print('='*84); print('Gate A — sharpness 지표가 곡률을 재고 있는가'); print('='*84)
    print('기준: 대칭/단측 비율 평균 ≥ 0.30 이고, 대칭이 |g| 와 완전상관(>0.95)이 아닐 것\n')
    rows = [(a['arm'], a['results']['final_one_sided'], a['results']['final_symmetric'],
             a['results']['final_probe_grad_norm']) for a in arms
            if a['results'].get('final_symmetric') is not None]
    if not rows: print('판정 불가: 대칭 sharpness 기록 없음'); return None
    print(f'{"arm":<30}{"one":>10}{"sym":>12}{"|g|":>9}{"sym/one":>9}')
    print('-'*70)
    for n,o,s,g in rows:
        print(f'{n:<30}{o:>10.4f}{s:>12.6f}{g:>9.3f}{s/o if o else float("nan"):>9.3f}')
    print('-'*70)
    ratios = [s/o for _,o,s,_ in rows if o]
    r_sg = pearson([s for _,_,s,_ in rows], [g for _,_,_,g in rows])
    r_1st = pearson([o-s for _,o,s,_ in rows], [0.05*g for _,_,_,g in rows])
    print(f'\n대칭/단측 비율 평균 : {mean(ratios):.3f} (sd {sd(ratios):.3f})')
    print(f'(one−sym) ~ ρ|g|    : {r_1st:+.3f}   ← 이론대로면 +1 근처')
    print(f'sym ~ |g|           : {r_sg:+.3f}   ← +1 근처면 곡률도 |g| 대리물')
    ok = mean(ratios) >= 0.30 and not abs(r_sg) > 0.95
    print('\n▶ ' + ('통과. 대칭 sharpness 를 주 지표로 승격.' if ok else
          '실패 → 사전등록 대응: "기존 지표는 곡률이 아니라 grad norm 의 대리물이었다"로\n'
          '   프레이밍 전환. 방법론적 기여가 주 결과가 된다.'))
    print()
    return {'passed': ok, 'ratio_mean': mean(ratios),
            'corr_sym_gradnorm': r_sg, 'corr_first_order': r_1st}

def gate_b(arms, rho_ref=0.05, metric='final_symmetric'):
    print('='*84); print(f'Gate B — sharpness 감소가 시드에 걸쳐 일관적인가 ({metric})')
    print('='*84)
    print('기준: 5/5 → p=0.031 통과 | 4/5 → 경향만 기술 | 3/5 이하 → 발견 철회\n')
    by = defaultdict(dict)
    for a in arms: by[a['seed']][a['rho_max']] = a['results']
    pairs = [(s, d[0.0][metric], d[rho_ref][metric],
              d[0.0]['final_test_acc'], d[rho_ref]['final_test_acc'])
             for s, d in sorted(by.items()) if 0.0 in d and rho_ref in d]
    if not pairs: print('판정 불가: 짝이 맞는 시드 없음'); return None
    print(f'{"seed":>6}{"off":>13}{"on":>13}{"Δsharp":>13}{"Δacc":>9}')
    print('-'*54)
    for s,so,sn,ao,an in pairs:
        print(f'{s:>6}{so:>13.6f}{sn:>13.6f}{sn-so:>13.6f}{an-ao:>+9.2f}')
    print('-'*54)
    n = len(pairs); k = sum(1 for _,so,sn,_,_ in pairs if sn < so)
    p = sign_test_p(k, n)
    ds = [sn-so for _,so,sn,_,_ in pairs]; da = [an-ao for _,_,_,ao,an in pairs]
    print(f'\nsam_on 이 더 낮음: {k}/{n}  부호검정 p = {p:.4f}')
    print(f'Δsharpness {mean(ds):+.6f} ± {sd(ds):.6f}')
    print(f'Δaccuracy  {mean(da):+.3f} ± {sd(da):.3f} %p')
    if n < 3:
        v = 'insufficient'
        print(f'\n▶ 판정 불가 — 시드가 {n}개뿐이다. 부호검정은 5시드부터 p=0.031 이 된다.')
        print('   (3시드 p=0.125, 4시드 p=0.0625 — 어느 쪽도 0.05 를 넘지 못한다)')
    elif k == n and n >= 5:
        v = 'pass'; print(f'\n▶ 통과. "SAM 은 sharpness 를 일관되게 낮춘다 ({n}/{n}, p={p:.3f})"')
    elif k >= n-1:
        v = 'partial'; print('\n▶ 부분 통과. "경향은 있으나 유의하지 않다"로 약화 기술, 효과크기 제시.')
    else:
        v = 'fail'; print('\n▶ 실패 → 사전등록 대응: "관찰된 감소는 시드 변동으로 설명된다".\n'
                          '   §5.3~5.5 재작성, 축을 SWA 라인으로 이동.')
    print()
    return {'verdict': v, 'n': n, 'n_lower': k, 'p': p,
            'd_sharp_mean': mean(ds), 'd_acc_mean': mean(da)}

def gate_c(arms, metric='final_symmetric'):
    print('='*84); print('Gate C — rho 반응이 sharpness 와 accuracy 에서 갈리는가'); print('='*84)
    print('기준: sharpness 는 ρ에 단조 감소, accuracy 는 평평(변동폭 ≤ 2×시드 sd) → 통과\n')
    by = defaultdict(list)
    for a in arms: by[a['rho_max']].append(a['results'])
    rhos = sorted(by)
    if len(rhos) < 3: print(f'판정 불가: rho 수준 {len(rhos)}개 (3개 이상 필요)'); return None
    print(f'{"rho":>6}{"n":>4}{"sharp":>14}{"±sd":>12}{"acc":>10}{"±sd":>9}')
    print('-'*55)
    sm, am, asd = [], [], []
    for r in rhos:
        s = [x[metric] for x in by[r]]; a = [x['final_test_acc'] for x in by[r]]
        sm.append(mean(s)); am.append(mean(a)); asd.append(sd(a))
        print(f'{r:>6.2f}{len(s):>4}{mean(s):>14.6f}{sd(s):>12.6f}{mean(a):>10.2f}{sd(a):>9.3f}')
    print('-'*55)
    mono = all(sm[i+1] <= sm[i] for i in range(len(sm)-1))
    spread = max(am) - min(am)
    band = max([x for x in asd if x == x] or [float('nan')])
    flat = band == band and spread <= 2*band
    print(f'\nsharpness ρ 단조 감소 : {"예" if mono else "아니오"}')
    print(f'accuracy 변동폭       : {spread:.3f} %p (시드 sd 최대 {band:.3f})')
    if mono and flat:
        v='pass'; print('\n▶ 통과 — 논문 메인 그림.\n'
                        '  "평탄화는 ρ에 단조 반응하지만 일반화는 그렇지 않다"')
    elif mono:
        v='partial'; print('\n▶ 부분 통과 (둘 다 단조) → "ρ에 비례해 이득" 실용 가이드로 전환')
    else:
        v='nonmono'; print(f'\n▶ 비단조 → "최적 ρ 가 존재한다" (최적 ≈ {rhos[sm.index(min(sm))]:g})')
    print()
    return {'verdict': v, 'rhos': rhos, 'sharp_mean': sm, 'acc_mean': am, 'acc_sd': asd}

print('게이트 함수 준비 완료')

In [ ]:
# ══ 9. 그리드 설정 ════════════════════════════════════════════════════════
# 처음에는 SEEDS=[42] 로 한 시드만 돌려 23분 시점(Stage 1 + 첫 arm)을 확인한다.
# 정상이면 [42,43,44,45,46] 으로 바꾸고 다시 실행 — 끝난 arm 은 건너뛴다.
SEEDS = [42]
RHOS  = [0.0, 0.02, 0.05, 0.10]          # 0.0 = sam_off

CFG = dict(stable_end=40, decay_epochs=20, warmup_epochs=3,
           total_epochs=60,                     # = stable_end + decay_epochs
           peak_lr=2e-4, weight_decay=0.5, grad_clip=1.0,
           batch_size=128, rho_ramp_epochs=3,
           sharpness_interval=2, sharpness_rho=0.05)

est = len(SEEDS) * (15.1 + 7.5 + 14.3*(len(RHOS)-1))
print(f'arm {len(SEEDS)*len(RHOS)}개 | 예상 {est:.0f}분 ({est/60:.1f}시간)')

In [ ]:
# ══ 10. 실행 ══════════════════════════════════════════════════════════════
# 위 그래프가 매 에폭 갱신된다.
for seed in SEEDS:
    ck = stage1(seed, CFG)
    for rho in RHOS:
        stage2(seed, rho, CFG, ck)
print('\n그리드 완료')

In [ ]:
# ══ 11. 결과 표 ═══════════════════════════════════════════════════════════
arms = load_arms()
print(f'{"seed":>5}{"rho":>7}{"acc0":>8}{"acc1":>8}{"one":>9}{"sym":>12}{"|g|":>8}{"min":>7}')
print('-'*64)
for a in sorted(arms, key=lambda x: (x['seed'], x['rho_max'])):
    r = a['results']
    print(f'{a["seed"]:>5}{a["rho_max"]:>7.2f}{r["decay_start_test_acc"]:>8.2f}'
          f'{r["final_test_acc"]:>8.2f}{r["final_one_sided"]:>9.4f}'
          f'{r["final_symmetric"]:>12.6f}{r["final_probe_grad_norm"]:>8.3f}'
          f'{r["decay_time_min"]:>7.1f}')
print('-'*64)
print('one = 단측 sharpness | sym = 대칭(곡률) | |g| = probe grad norm')

In [ ]:
# ══ 12. Gate A / B / C 판정 ═══════════════════════════════════════════════
arms = load_arms()
res = {'gate_a': gate_a(arms), 'gate_b': gate_b(arms), 'gate_c': gate_c(arms)}
json.dump(res, open(os.path.join(OUT_DIR, '_gate_results.json'), 'w',
                    encoding='utf-8'), ensure_ascii=False, indent=2)

In [ ]:
# ══ 13. rho 스윕 그림 (Gate C) ════════════════════════════════════════════
by = defaultdict(lambda: {'sym': [], 'acc': []})
for a in load_arms():
    by[a['rho_max']]['sym'].append(a['results']['final_symmetric'])
    by[a['rho_max']]['acc'].append(a['results']['final_test_acc'])
rhos = sorted(by)
if len(rhos) >= 2:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].errorbar(rhos, [mean(by[r]['sym']) for r in rhos],
                   yerr=[sd(by[r]['sym']) if len(by[r]['sym'])>1 else 0 for r in rhos],
                   marker='o', capsize=4, color='tab:purple')
    ax[0].set_xlabel('rho'); ax[0].set_ylabel('symmetric sharpness')
    ax[0].set_title('Flatness responds to rho'); ax[0].grid(alpha=.3)
    ax[1].errorbar(rhos, [mean(by[r]['acc']) for r in rhos],
                   yerr=[sd(by[r]['acc']) if len(by[r]['acc'])>1 else 0 for r in rhos],
                   marker='o', capsize=4, color='tab:blue')
    ax[1].set_xlabel('rho'); ax[1].set_ylabel('test accuracy (%)')
    ax[1].set_title('Generalization does not'); ax[1].grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'fig_rho_sweep.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('rho 수준이 부족하다. 그리드를 완주할 것.')

In [ ]:
# ══ ★ 회수 (세션 끄기 전 반드시 실행) ═════════════════════════════════════
# Colab VM 은 임시다. Kaggle 에서 이미 한 번 전부 날렸다 (PROGRESS.md §8).
arc = os.path.join(OUT_DIR, 'phase1_results.tar.gz')
with tarfile.open(arc, 'w:gz') as t:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.json'))
                    + glob.glob(os.path.join(OUT_DIR, '*.png'))):
        t.add(f, arcname=os.path.basename(f))
print(f'{arc}  ({os.path.getsize(arc)/1024:.0f} KB)')
try:
    from google.colab import files; files.download(arc)
except Exception as e:
    print('브라우저 다운로드 불가:', e)
    print('CLI 로:  colab download -s <세션>', arc, './results/')

---

# ==============================================================================
# 섹션 2 — Phase 3 — Muon 라인 마무리
# ==============================================================================

GPU 필요. 약 1시간. 우선순위 낮음.

원본: `3_muon.ipynb`  ·  셀 12개

# Phase 3 — Muon 라인 마무리 (약 1시간, 자투리 시간용)

Muon 을 더 튜닝하는 게 아니다. `docs/PROGRESS.md` §5.2 는 이미 결론이 났다
(전 구성이 SAM 베이스라인 94.69 % 에 7~9 %p 미달).

남기려는 문장 하나 — **"SAM 의 sharpness 감소는 옵티마이저와 무관하게 작동한다"** —
가 노트북 07(Muon v3) 수치에 기대고 있는데, 그 값들은 probe 가 MODE 마다
재생성되던 시절 것이라 **상호 비교가 불가능하다**(§6.3).
노트북 08(v4)은 고정 probe 를 쓰지만 `muon_only` 만 돌렸다.

여기서는 v4 와 같은 조건으로 `muon_only` 와 `sam_always` 를 나란히 돌려
짝이 맞는 비교를 만든다. 우선순위는 낮다 — Phase 1 이 GPU 를 놀리는 틈에 끼우면 된다.

In [ ]:
# ══ 1. 셋업 ═══════════════════════════════════════════════════════════════
import os, sys, math, json, time, glob, random, tarfile, urllib.request
from datetime import datetime
from collections import defaultdict

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ── 결과 저장 위치 ────────────────────────────────────────────────────────
# /content 는 세션이 끊기면 사라진다. 어느 쪽으로 돌리느냐에 따라 다르다.
#
#   CLI (colab exec)  : 기본값 그대로. 세션을 살려두고 끝나면 colab download 로 회수.
#                       drive.mount 는 대화형이라 헤드리스에서 멈출 수 있으므로 쓰지 않는다.
#   Colab 웹에서 5시간 : USE_DRIVE = True 로 두면 중단돼도 이어서 갈 수 있다.
USE_DRIVE = False

OUT_DIR = '/content/out'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/wsd_sam/out'
    except Exception as e:
        print('Drive 마운트 실패 → /content/out 사용:', e)

DATA_DIR = os.environ.get('DATA_DIR', '/content/data')
OUT_DIR  = os.environ.get('OUT_DIR', OUT_DIR)
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device,
      f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else '')
print('torch  :', torch.__version__)
print('출력   :', OUT_DIR)
if device.type != 'cuda':
    print('\n⚠️ GPU 가 아니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿀 것.')

In [ ]:
# ══ 2. 재현성 ═════════════════════════════════════════════════════════════
# 노트북판의 가장 큰 결함이 "Stage 2 에 시드가 없다"는 것이었다.
# sam_off 셀이 RNG 를 소모한 뒤 sam_on 셀이 시작돼, 두 arm 이 서로 다른
# 데이터 순서를 봤다. 아래 세 함수로 그 경로를 원천 차단한다.
def seed_everything(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

def worker_init(worker_id):
    random.seed(torch.initial_seed() % 2**32 + worker_id)

print('시드 유틸 준비 완료')

In [ ]:
# ══ 3. 데이터 ═════════════════════════════════════════════════════════════
CIFAR_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
C10_MEAN, C10_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)

def prepare_data(data_dir=DATA_DIR):
    root = os.path.join(data_dir, 'cifar10')
    if not os.path.exists(root):
        tgz = os.path.join(data_dir, 'cifar10.tgz')
        if not os.path.exists(tgz):
            print('CIFAR-10 다운로드 중...', flush=True)
            urllib.request.urlretrieve(CIFAR_URL, tgz)
        print('압축 해제 중...', flush=True)
        with tarfile.open(tgz, 'r:gz') as tar:
            tar.extractall(path=data_dir, filter='data')
    return root

def build_loaders(seed, batch_size=128, num_workers=2):
    root = prepare_data()
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    tf_eval = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    train = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_train)
    test  = datasets.ImageFolder(os.path.join(root, 'test'),  transform=tf_eval)
    # probe 전용: 같은 train 이미지지만 augmentation 없이 결정론적으로 읽는다
    probe_src = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_eval)
    tl = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=True, generator=loader_generator(seed),
                    worker_init_fn=worker_init)
    vl = DataLoader(test, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tl, vl, probe_src

def make_probe(probe_src, n=384, batch_size=128, probe_seed=1234):
    """sharpness 측정용 고정 배치.
    인덱스는 별도 Generator 로 뽑고 이미지는 augmentation 이 없으므로,
    세션이 달라도 항상 같은 텐서가 나온다."""
    g = torch.Generator().manual_seed(probe_seed)
    idx = torch.randperm(len(probe_src), generator=g)[:n].tolist()
    loader = DataLoader(Subset(probe_src, idx), batch_size=batch_size, shuffle=False)
    return [(x.to(device), y.to(device)) for x, y in loader]

print('데이터 함수 준비 완료')

In [ ]:
# ══ 4. 옵티마이저 + SAM ═══════════════════════════════════════════════════
class Lion(optim.Optimizer):
    """update = sign(b1*m + (1-b1)*g),  m = b2*m + (1-b2)*g"""
    def __init__(self, params, lr=2e-4, betas=(0.9, 0.99), weight_decay=0.5):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            b1, b2 = group['betas']
            for p in group['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'exp_avg' not in st: st['exp_avg'] = torch.zeros_like(p)
                m, g = st['exp_avg'], p.grad
                if wd: p.mul_(1 - lr * wd)
                p.add_(m.mul(b1).add_(g, alpha=1-b1).sign_(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1-b2)


class SAMAscent:
    """SAM 을 '옵티마이저'가 아니라 '그래디언트 변환기'로 분리한 것.
    perturbation 만 담당하고 step 은 외부 옵티마이저가 하므로,
    옵티마이저를 교체하지 않고 SAM 을 켜고 끌 수 있다 → 모멘텀이 연속된다."""
    def __init__(self, params, rho=0.0):
        self.params = [p for p in params if p.requires_grad]
        self.rho = rho; self._e_w = {}

    def set_rho(self, rho): self.rho = rho

    @torch.no_grad()
    def grad_norm(self):
        gs = [p.grad.norm(p=2) for p in self.params if p.grad is not None]
        return torch.norm(torch.stack(gs), p=2) if gs else torch.tensor(0.0)

    @torch.no_grad()
    def ascent(self):
        if self.rho <= 0: return False
        gn = self.grad_norm()
        if gn.item() < 1e-12: return False
        scale = self.rho / (gn + 1e-12)
        self._e_w.clear()
        for p in self.params:
            if p.grad is None: continue
            e = p.grad * scale.to(p.device); p.add_(e); self._e_w[p] = e
        return True

    @torch.no_grad()
    def restore(self):
        for p, e in self._e_w.items(): p.sub_(e)
        self._e_w.clear()

def build_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)

print('Lion / SAMAscent / ResNet-18 준비 완료')

In [ ]:
# ══ 5. WSD 스케줄 + sharpness 측정 ════════════════════════════════════════
def wsd_lr(epoch, total_epochs, peak_lr, warmup_epochs, decay_epochs):
    """warmup(선형 상승) → stable(유지) → decay(선형 하강)"""
    stable_end = total_epochs - decay_epochs
    if epoch < warmup_epochs:
        return peak_lr * (epoch + 1) / warmup_epochs
    if epoch < stable_end:
        return peak_lr
    prog = min(1.0, max(0.0, (epoch - stable_end + 1) / decay_epochs))
    return peak_lr * (1.0 - prog)


@torch.no_grad()
def _perturb(e_ws, sign):
    for p, e in e_ws: p.add_(e, alpha=sign)


def measure_sharpness(model, criterion, probe, rho=0.05):
    """세 값을 함께 반환한다.

    one_sided : L(w+e) − L(w)
        테일러 전개하면 ≈ rho*||g|| + (1/2)rho^2 * (g'Hg)/||g||^2 이다.
        즉 1차항이 그래디언트 노름이라, 이 값은 곡률이 아니라 ||g|| 를
        상당 부분 재고 있다. 과거 결과와 비교하려고 유지한다.

    symmetric : (1/2)(L(w+e) + L(w−e)) − L(w)
        1차항이 상쇄되어 곡률 항만 남는다. 이쪽이 우리가 재려던 것.

    grad_norm : 위 해석을 사후 검증하기 위한 원자료 (Gate A).
    """
    was_training = model.training
    model.eval()
    one, sym, gns = [], [], []
    for x, y in probe:
        with torch.no_grad():
            l0 = criterion(model(x), y).item()
        model.zero_grad(set_to_none=True)
        criterion(model(x), y).backward()
        gs = [p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]
        gn = torch.norm(torch.stack(gs), p=2); gns.append(gn.item())
        scale = rho / (gn + 1e-12)
        e_ws = [(p, p.grad * scale) for p in model.parameters() if p.grad is not None]
        with torch.no_grad():
            _perturb(e_ws, +1.0);  lp = criterion(model(x), y).item()
            _perturb(e_ws, -2.0);  lm = criterion(model(x), y).item()
            _perturb(e_ws, +1.0)                     # 원위치 복원
        model.zero_grad(set_to_none=True)
        one.append(lp - l0); sym.append(0.5 * (lp + lm) - l0)
    if was_training: model.train()
    n = len(one)
    return sum(one)/n, sum(sym)/n, sum(gns)/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        total += y.size(0); correct += model(x).max(1)[1].eq(y).sum().item()
    return 100.0 * correct / total

print('스케줄 / sharpness / 평가 함수 준비 완료')

In [ ]:
# ══ 6. 실시간 그래프 ══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 기본값은 꺼둔다. 켜면 clear_output 으로 화면을 새로 그리기 때문에
# 에폭 로그가 쌓이지 않고 매번 최신 한 줄만 남는다.
# 로그를 한 줄씩 쌓아 보려면 그대로 두고, 그래프가 필요하면 'on': True 로.
LIVE = {'on': False, 'every': 1}

def live_plot(tag, history, sharp):
    if not LIVE['on'] or len(history['epoch']) % LIVE['every']: return
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history['epoch'], history['train_acc'], lw=1.2, label='train')
    ax[0].plot(history['epoch'], history['test_acc'],  lw=1.6, label='test')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
    ax[0].set_title(f'{tag} — test {history["test_acc"][-1]:.2f}%')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if sharp['epoch']:
        ax[1].plot(sharp['epoch'], sharp['one_sided'], 'o-', ms=3,
                   color='tab:orange', label='one-sided')
        ax[1].plot(sharp['epoch'], sharp['symmetric'], 's-', ms=3,
                   color='tab:purple', label='symmetric (curvature)')
        ax[1].set_xlabel('epoch'); ax[1].set_ylabel('sharpness')
        ax[1].set_title('sharpness (lower = flatter)')
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(history['epoch'], history['grad_norm'], color='tab:brown', lw=1.2)
    ax[2].set_xlabel('epoch'); ax[2].set_ylabel('grad norm', color='tab:brown')
    ax[2].grid(alpha=.3); ax[2].set_title('grad norm & rho')
    a2 = ax[2].twinx(); a2.plot(history['epoch'], history['rho'],
                                color='tab:green', lw=1.4)
    a2.set_ylabel('rho', color='tab:green')
    plt.tight_layout(); plt.show()
    print(f'[{tag}] epoch {history["epoch"][-1]} | '
          f'loss {history["train_loss"][-1]:.4f} | '
          f'test {history["test_acc"][-1]:.2f}% | '
          f'{sum(history["time"])/60:.1f}분 경과', flush=True)

print('그래프 훅 준비 완료 — 기본 꺼짐. 켜려면 LIVE["on"] = True')

In [ ]:
# ══ Muon 옵티마이저 ═══════════════════════════════════════════════════════
@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Newton-Schulz 반복으로 gradient 를 직교화한다 (Muon 의 핵심 연산)."""
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16(); X /= (X.norm() + eps)
    t = G.size(0) > G.size(1)
    if t: X = X.T
    for _ in range(steps):
        A = X @ X.T; B = b*A + c*A@A; X = a*X + B@X
    if t: X = X.T
    return X.to(G.dtype)


class Muon(optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      ns_steps=ns_steps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                g = p.grad
                if g is None: continue
                st = self.state[p]
                if 'momentum_buffer' not in st: st['momentum_buffer'] = torch.zeros_like(g)
                buf = st['momentum_buffer']
                shape = g.shape
                if g.ndim > 2: g = g.view(g.size(0), -1)
                go = zeropower_via_newtonschulz5(g, steps=group['ns_steps']).view(shape)
                buf.mul_(group['momentum']).add_(go)
                upd = go + group['momentum']*buf if group['nesterov'] else buf
                if group['weight_decay']: p.mul_(1 - group['lr']*group['weight_decay'])
                p.add_(upd, alpha=-group['lr'])


def split_params(model, scope='hidden_only', verbose=True):
    """'hidden_only' = Muon 공식 가이드. 분류기 헤드(fc)와 stem conv(conv1)를 제외.
    노트북 08(v4)과 같은 조건이다."""
    muon_p, adamw_p, excl = [], [], []
    for name, p in model.named_parameters():
        if p.ndim < 2: adamw_p.append(p); continue
        if scope == 'hidden_only' and (name.startswith('fc.') or name.startswith('conv1.')):
            adamw_p.append(p); excl.append((name, tuple(p.shape))); continue
        muon_p.append(p)
    if verbose:
        print(f"scope='{scope}'  Muon {len(muon_p)}텐서/"
              f"{sum(p.numel() for p in muon_p):,}  AdamW {len(adamw_p)}텐서/"
              f"{sum(p.numel() for p in adamw_p):,}")
        for n, s in excl: print(f'   제외: {n} {s}')
    return muon_p, adamw_p

print('Muon 준비 완료')

In [ ]:
# ══ 실행 함수 ═════════════════════════════════════════════════════════════
def run_muon(mode, seed=42, epochs=50, scope='hidden_only', rho_max=0.05,
             ramp_epochs=5, muon_lr=0.02, muon_wd=0.0, adamw_lr=3e-4,
             grad_clip=1.0, sharpness_interval=2, sharpness_rho=0.05,
             batch_size=128):
    """mode: 'muon_only' | 'sam_always'"""
    out = os.path.join(OUT_DIR, f'muon_{mode}_{scope}_seed{seed}.json')
    if os.path.exists(out):
        print(f'skip {mode} (이미 있음)'); return json.load(open(out, encoding='utf-8'))

    seed_everything(seed)
    tl, vl, ps = build_loaders(seed, batch_size)
    probe = make_probe(ps)
    model = build_model()
    mp, ap = split_params(model, scope)
    om = Muon(mp, lr=muon_lr, momentum=0.95, weight_decay=muon_wd)
    oa = optim.AdamW(ap, lr=adamw_lr, weight_decay=0.01)
    sm = optim.lr_scheduler.CosineAnnealingLR(om, T_max=epochs)
    sa = optim.lr_scheduler.CosineAnnealingLR(oa, T_max=epochs)
    sam = SAMAscent(model.parameters()); crit = nn.CrossEntropyLoss()

    hist = {k: [] for k in ['epoch','phase','lr','rho','train_loss','train_acc',
                            'test_acc','grad_norm','ascent_gap','time']}
    sharp = {'epoch': [], 'one_sided': [], 'symmetric': [], 'probe_grad_norm': []}

    print(f'\n=== Muon [{mode} / {scope} / seed {seed}] {epochs} epochs ===\n', flush=True)
    for epoch in range(epochs):
        rho_now = (rho_max * min(1.0, (epoch+1)/max(1, ramp_epochs))
                   if mode == 'sam_always' else 0.0)
        sam.set_rho(rho_now)
        model.train()
        rl = rg = rn = 0.0; correct = total = nb = 0; t0 = time.time()
        for x, y in tl:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            om.zero_grad(set_to_none=True); oa.zero_grad(set_to_none=True)
            o = model(x); loss = crit(o, y); loss.backward()
            total += y.size(0); correct += o.max(1)[1].eq(y).sum().item()
            rn += sam.grad_norm().item()
            gap = 0.0
            if sam.ascent():
                om.zero_grad(set_to_none=True); oa.zero_grad(set_to_none=True)
                la = crit(model(x), y); la.backward(); gap = la.item()-loss.item()
                sam.restore()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            om.step(); oa.step(); rl += loss.item(); rg += gap; nb += 1
        sm.step(); sa.step()
        dt = time.time()-t0
        te = evaluate(model, vl)
        for k, v in [('epoch',epoch+1),('phase',mode),('lr',om.param_groups[0]['lr']),
                     ('rho',rho_now),('train_loss',rl/nb),('train_acc',100.0*correct/total),
                     ('test_acc',te),('grad_norm',rn/nb),('ascent_gap',rg/nb),('time',dt)]:
            hist[k].append(v)
        extra = ''
        if (epoch+1) % sharpness_interval == 0 or epoch == epochs-1:
            o_, s_, g_ = measure_sharpness(model, crit, probe, sharpness_rho)
            sharp['epoch'].append(epoch+1); sharp['one_sided'].append(o_)
            sharp['symmetric'].append(s_); sharp['probe_grad_norm'].append(g_)
            extra = f' | Sharp1 {o_:.4f} | SharpSym {s_:.6f} | pg {g_:.4f}'
        print(f'[{mode}] Ep [{epoch+1}/{epochs}] | {dt:.1f}s | Loss {rl/nb:.4f} | '
              f'Train {100.0*correct/total:.2f}% | Test {te:.2f}% | '
              f'rho {rho_now:.4f} | gn {rn/nb:.3f}{extra}', flush=True)
        try: live_plot(mode, hist, sharp)
        except Exception as e: print('  (그래프 오류 무시:', e, ')', flush=True)

    best = max(hist['test_acc'])
    rec = {'mode': mode, 'scope': scope, 'seed': seed, 'epochs': epochs,
           'rho_max': rho_max if mode == 'sam_always' else 0.0,
           'results': {'final_train_acc': hist['train_acc'][-1],
                       'final_test_acc': hist['test_acc'][-1], 'best_test_acc': best,
                       'final_gap': hist['train_acc'][-1]-hist['test_acc'][-1],
                       'final_one_sided': sharp['one_sided'][-1],
                       'final_symmetric': sharp['symmetric'][-1],
                       'final_probe_grad_norm': sharp['probe_grad_norm'][-1],
                       'total_time_min': sum(hist['time'])/60},
           'history': hist, 'sharpness_history': sharp,
           'timestamp': datetime.now().isoformat(timespec='seconds')}
    json.dump(rec, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    print(f'\n저장: {out}', flush=True)
    return rec

print('run_muon 준비 완료')

In [ ]:
# ══ 실행 (두 MODE 연속) ═══════════════════════════════════════════════════
EPOCHS = 50          # 스모크로 먼저 보려면 2 로 낮출 것
SEED   = 42

recs = {m: run_muon(m, seed=SEED, epochs=EPOCHS) for m in ['muon_only', 'sam_always']}

In [ ]:
# ══ 비교 — 이 표 한 장이 §5.2 의 한 문단을 대체한다 ═══════════════════════
V4_REF = {'final_test': 85.29, 'one_sided': 0.3738}   # 노트북 08 실측
print(f'{"mode":<14}{"Test":>9}{"Best":>9}{"Sharp1":>10}{"SharpSym":>12}{"|g|":>9}{"min":>7}')
print('-'*70)
for m, rec in recs.items():
    r = rec['results']
    print(f'{m:<14}{r["final_test_acc"]:>9.2f}{r["best_test_acc"]:>9.2f}'
          f'{r["final_one_sided"]:>10.4f}{r["final_symmetric"]:>12.6f}'
          f'{r["final_probe_grad_norm"]:>9.3f}{r["total_time_min"]:>7.1f}')
print('-'*70)
print(f'참고) 노트북 08(v4) muon_only: Test {V4_REF["final_test"]:.2f}% / '
      f'Sharp1 {V4_REF["one_sided"]:.4f}')

a, b = recs['muon_only']['results'], recs['sam_always']['results']
d_sym = b['final_symmetric'] - a['final_symmetric']
print(f'\nSAM 효과: SharpSym {d_sym:+.6f} | '
      f'Test {b["final_test_acc"]-a["final_test_acc"]:+.2f}%p')
print('→ ' + ('Muon 에서도 SAM 이 곡률을 낮춘다. "옵티마이저 무관"을 측정값으로 쓸 수 있다.'
               if d_sym < 0 else
               'Muon 에서는 SAM 이 곡률을 낮추지 못했다. 해당 문장을 Lion 한정으로 축소할 것.'))

In [ ]:
# ══ ★ 회수 (세션 끄기 전 반드시 실행) ═════════════════════════════════════
# Colab VM 은 임시다. Kaggle 에서 이미 한 번 전부 날렸다 (PROGRESS.md §8).
arc = os.path.join(OUT_DIR, 'phase3_muon_results.tar.gz')
with tarfile.open(arc, 'w:gz') as t:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.json'))
                    + glob.glob(os.path.join(OUT_DIR, '*.png'))):
        t.add(f, arcname=os.path.basename(f))
print(f'{arc}  ({os.path.getsize(arc)/1024:.0f} KB)')
try:
    from google.colab import files; files.download(arc)
except Exception as e:
    print('브라우저 다운로드 불가:', e)
    print('CLI 로:  colab download -s <세션>', arc, './results/')

---

# ==============================================================================
# 섹션 3 — Phase 2 — ViT-Tiny 재현
# ==============================================================================

GPU 필요. Stage P 36분 + 시드당 30분. Phase 1 게이트 통과 후에.

원본: `2_vit.ipynb`  ·  셀 13개

# Phase 2 — ViT-Tiny 재현

노트북 11 의 설계를 그대로 옮기되 `1_wsd_grid.ipynb` 와 **같은 교정**을 적용했다
(Stage 2 시드 고정 · probe 분리 · 대칭 sharpness).

**Phase 1 이 Gate B/C 를 통과한 뒤에 돌린다.**
ResNet 결과가 애매하면 "두 아키텍처에서 재현"이라는 논지 자체를 다시 써야 하므로,
크로스-아키텍처 확장 전에 메인 축을 확정하는 게 안전하다.

| | |
|---|---|
| Stage P (사전학습) | 36분 — **한 번만.** 모든 시드가 이 backbone 을 공유 |
| 시드당 | Stage 1 12분 + arm 당 6~12분 |
| 3시드 | 약 2.1시간 |

⚠️ ViT 는 `label_smoothing=0.1` + 64×64 + 파라미터 수가 달라
**sharpness 절대값을 ResNet 과 같은 축에 얹으면 안 된다.** 방향·상대비로만 쓴다.

> ⚠️ 마지막 셀에서 결과와 **사전학습 체크포인트를 반드시 회수**할 것.
> 다시 만들면 backbone 이 달라져 기존 수치와 비교가 끊긴다.

In [ ]:
# ══ 1. 셋업 ═══════════════════════════════════════════════════════════════
import os, sys, math, json, time, glob, random, tarfile, urllib.request
from datetime import datetime
from collections import defaultdict

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ── 결과 저장 위치 ────────────────────────────────────────────────────────
# /content 는 세션이 끊기면 사라진다. 어느 쪽으로 돌리느냐에 따라 다르다.
#
#   CLI (colab exec)  : 기본값 그대로. 세션을 살려두고 끝나면 colab download 로 회수.
#                       drive.mount 는 대화형이라 헤드리스에서 멈출 수 있으므로 쓰지 않는다.
#   Colab 웹에서 5시간 : USE_DRIVE = True 로 두면 중단돼도 이어서 갈 수 있다.
USE_DRIVE = False

OUT_DIR = '/content/out'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/wsd_sam/out'
    except Exception as e:
        print('Drive 마운트 실패 → /content/out 사용:', e)

DATA_DIR = os.environ.get('DATA_DIR', '/content/data')
OUT_DIR  = os.environ.get('OUT_DIR', OUT_DIR)
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device,
      f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else '')
print('torch  :', torch.__version__)
print('출력   :', OUT_DIR)
if device.type != 'cuda':
    print('\n⚠️ GPU 가 아니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿀 것.')

In [ ]:
# ══ 2. 재현성 ═════════════════════════════════════════════════════════════
# 노트북판의 가장 큰 결함이 "Stage 2 에 시드가 없다"는 것이었다.
# sam_off 셀이 RNG 를 소모한 뒤 sam_on 셀이 시작돼, 두 arm 이 서로 다른
# 데이터 순서를 봤다. 아래 세 함수로 그 경로를 원천 차단한다.
def seed_everything(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

def worker_init(worker_id):
    random.seed(torch.initial_seed() % 2**32 + worker_id)

print('시드 유틸 준비 완료')

In [ ]:
# ══ 3. 데이터 (Tiny-ImageNet + CIFAR-10) ═════════════════════════════════
from PIL import Image
from torch.utils.data import Dataset
import zipfile

TIN_URL = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
CIFAR_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
TIN_MEAN, TIN_STD = (0.4802, 0.4481, 0.3975), (0.2770, 0.2691, 0.2821)
C10_MEAN, C10_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
IMG_SIZE = 64        # ViT 입력. CIFAR-10(32)도 여기 맞춰 리사이즈해야
                     # 사전학습 backbone(patch_embed, pos_embed)을 그대로 재사용한다


class TinyImageNetVal(Dataset):
    """val 은 클래스별로 나뉘어 있지 않고 val_annotations.txt 에 매핑이 있다."""
    def __init__(self, root, class_to_idx, transform=None):
        self.transform = transform; self.samples = []
        with open(os.path.join(root, 'val', 'val_annotations.txt')) as f:
            for line in f:
                q = line.strip().split('\t')
                if len(q) >= 2 and q[1] in class_to_idx:
                    self.samples.append((os.path.join(root,'val','images',q[0]),
                                         class_to_idx[q[1]]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert('RGB')      # 흑백 이미지가 섞여 있다
        return (self.transform(img) if self.transform else img), y


def make_transforms(mean, std, train, strong_aug):
    if not train:
        return transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE)),
           transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE//8),
           transforms.RandomHorizontalFlip()]
    if strong_aug:
        # ViT 는 inductive bias 가 없어 강한 증강이 사실상 필수 (DeiT 레시피)
        ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
    ops += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(ops)


def prepare_tin():
    root = os.path.join(DATA_DIR, 'tiny-imagenet-200')
    if not os.path.exists(root):
        z = os.path.join(DATA_DIR, 'tiny-imagenet-200.zip')
        if not os.path.exists(z):
            print('Tiny-ImageNet 다운로드 중... (248MB)', flush=True)
            urllib.request.urlretrieve(TIN_URL, z)
        print('압축 해제 중...', flush=True)
        with zipfile.ZipFile(z) as f: f.extractall(DATA_DIR)
    return root


def prepare_cifar():
    root = os.path.join(DATA_DIR, 'cifar10')
    if not os.path.exists(root):
        t = os.path.join(DATA_DIR, 'cifar10.tgz')
        if not os.path.exists(t):
            print('CIFAR-10 다운로드 중...', flush=True)
            urllib.request.urlretrieve(CIFAR_URL, t)
        with tarfile.open(t, 'r:gz') as tar: tar.extractall(path=DATA_DIR, filter='data')
    return root


def make_probe(dataset, n=384, batch_size=128, probe_seed=1234):
    g = torch.Generator().manual_seed(probe_seed)
    idx = torch.randperm(len(dataset), generator=g)[:n].tolist()
    return [(x.to(device), y.to(device))
            for x, y in DataLoader(Subset(dataset, idx), batch_size=batch_size)]


def cifar_loaders(seed, batch_size=256, num_workers=2):
    root = prepare_cifar()
    tr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(C10_MEAN, C10_STD, True, False))
    te = datasets.ImageFolder(os.path.join(root,'test'),
                              transform=make_transforms(C10_MEAN, C10_STD, False, False))
    pr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(C10_MEAN, C10_STD, False, False))
    tl = DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=True, drop_last=True,
                    generator=loader_generator(seed), worker_init_fn=worker_init)
    vl = DataLoader(te, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tl, vl, pr

print('데이터 함수 준비 완료')

In [ ]:
# ══ 4. ViT-Tiny + Lion + SAM ══════════════════════════════════════════════
class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch=8, in_ch=3, dim=192):
        super().__init__()
        self.num_patches = (img_size//patch)**2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        h = int(dim*mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim,h), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(h,dim), nn.Dropout(drop))
    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):
    """ViT-Tiny 급. 64x64 / patch 8 → 64 토큰 + cls = 65"""
    def __init__(self, num_classes=200, img_size=64, patch=8, dim=192,
                 depth=6, heads=3, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch, 3, dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1,1,dim))
        self.pos_embed = nn.Parameter(torch.zeros(1,n+1,dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([Block(dim,heads,mlp_ratio,drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim); self.head = nn.Linear(dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], 1) + self.pos_embed
        x = self.pos_drop(x)
        for b in self.blocks: x = b(x)
        return self.head(self.norm(x)[:, 0])

def build_vit(num_classes): return ViT(num_classes=num_classes).to(device)

def load_backbone(model, sd):
    """head 를 뺀 backbone 만 이식 (200클래스 → 10클래스 전이)"""
    bb = {k: v for k, v in sd.items() if not k.startswith('head.')}
    missing, _ = model.load_state_dict(bb, strict=False)
    print('   backbone 이식 — 새로 초기화:', missing, flush=True)
    return model


class Lion(optim.Optimizer):
    def __init__(self, params, lr=2e-4, betas=(0.9,0.99), weight_decay=0.5):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for g_ in self.param_groups:
            lr, wd = g_['lr'], g_['weight_decay']; b1, b2 = g_['betas']
            for p in g_['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'exp_avg' not in st: st['exp_avg'] = torch.zeros_like(p)
                m, g = st['exp_avg'], p.grad
                if wd: p.mul_(1 - lr*wd)
                p.add_(m.mul(b1).add_(g, alpha=1-b1).sign_(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1-b2)


class SAMAscent:
    def __init__(self, params, rho=0.0):
        self.params = [p for p in params if p.requires_grad]
        self.rho = rho; self._e_w = {}
    def set_rho(self, r): self.rho = r
    @torch.no_grad()
    def grad_norm(self):
        gs = [p.grad.norm(p=2) for p in self.params if p.grad is not None]
        return torch.norm(torch.stack(gs), p=2) if gs else torch.tensor(0.0)
    @torch.no_grad()
    def ascent(self):
        if self.rho <= 0: return False
        gn = self.grad_norm()
        if gn.item() < 1e-12: return False
        sc = self.rho/(gn+1e-12); self._e_w.clear()
        for p in self.params:
            if p.grad is None: continue
            e = p.grad*sc.to(p.device); p.add_(e); self._e_w[p] = e
        return True
    @torch.no_grad()
    def restore(self):
        for p, e in self._e_w.items(): p.sub_(e)
        self._e_w.clear()

print('ViT / Lion / SAMAscent 준비 완료')

In [ ]:
# ══ 5. 스케줄 + sharpness ═════════════════════════════════════════════════
def wsd_lr(epoch, total, peak, warm, decay):
    se = total - decay
    if epoch < warm: return peak*(epoch+1)/warm
    if epoch < se: return peak
    return peak*(1.0 - min(1.0, max(0.0, (epoch-se+1)/decay)))

@torch.no_grad()
def _perturb(e_ws, sign):
    for p, e in e_ws: p.add_(e, alpha=sign)

def measure_sharpness(model, criterion, probe, rho=0.05):
    """one_sided = L(w+e)−L(w)  ≈ rho*||g|| + 곡률   (1차항이 grad norm)
       symmetric = (L(w+e)+L(w−e))/2 − L(w)          (1차항 상쇄, 곡률만)
       grad_norm = Gate A 검증용 원자료"""
    was = model.training; model.eval()
    one, sym, gns = [], [], []
    for x, y in probe:
        with torch.no_grad(): l0 = criterion(model(x), y).item()
        model.zero_grad(set_to_none=True); criterion(model(x), y).backward()
        gs = [p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]
        gn = torch.norm(torch.stack(gs), p=2); gns.append(gn.item())
        sc = rho/(gn+1e-12)
        e_ws = [(p, p.grad*sc) for p in model.parameters() if p.grad is not None]
        with torch.no_grad():
            _perturb(e_ws, +1.0); lp = criterion(model(x), y).item()
            _perturb(e_ws, -2.0); lm = criterion(model(x), y).item()
            _perturb(e_ws, +1.0)
        model.zero_grad(set_to_none=True)
        one.append(lp-l0); sym.append(0.5*(lp+lm)-l0)
    if was: model.train()
    n = len(one)
    return sum(one)/n, sum(sym)/n, sum(gns)/n

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        t += y.size(0); c += model(x).max(1)[1].eq(y).sum().item()
    return 100.0*c/t

def new_history():
    keys = ['epoch','phase','lr','rho','train_loss','train_acc','test_acc',
            'grad_norm','ascent_gap','time']
    return ({k: [] for k in keys},
            {'epoch': [], 'one_sided': [], 'symmetric': [], 'probe_grad_norm': []})

print('스케줄 / sharpness 준비 완료')

In [ ]:
# ══ 6. 실시간 그래프 ══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 기본값은 꺼둔다. 켜면 clear_output 으로 화면을 새로 그리기 때문에
# 에폭 로그가 쌓이지 않고 매번 최신 한 줄만 남는다.
# 로그를 한 줄씩 쌓아 보려면 그대로 두고, 그래프가 필요하면 'on': True 로.
LIVE = {'on': False, 'every': 1}

def live_plot(tag, history, sharp):
    if not LIVE['on'] or len(history['epoch']) % LIVE['every']: return
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history['epoch'], history['train_acc'], lw=1.2, label='train')
    ax[0].plot(history['epoch'], history['test_acc'],  lw=1.6, label='test')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
    ax[0].set_title(f'{tag} — test {history["test_acc"][-1]:.2f}%')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if sharp['epoch']:
        ax[1].plot(sharp['epoch'], sharp['one_sided'], 'o-', ms=3,
                   color='tab:orange', label='one-sided')
        ax[1].plot(sharp['epoch'], sharp['symmetric'], 's-', ms=3,
                   color='tab:purple', label='symmetric (curvature)')
        ax[1].set_xlabel('epoch'); ax[1].set_ylabel('sharpness')
        ax[1].set_title('sharpness (lower = flatter)')
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(history['epoch'], history['grad_norm'], color='tab:brown', lw=1.2)
    ax[2].set_xlabel('epoch'); ax[2].set_ylabel('grad norm', color='tab:brown')
    ax[2].grid(alpha=.3); ax[2].set_title('grad norm & rho')
    a2 = ax[2].twinx(); a2.plot(history['epoch'], history['rho'],
                                color='tab:green', lw=1.4)
    a2.set_ylabel('rho', color='tab:green')
    plt.tight_layout(); plt.show()
    print(f'[{tag}] epoch {history["epoch"][-1]} | '
          f'loss {history["train_loss"][-1]:.4f} | '
          f'test {history["test_acc"][-1]:.2f}% | '
          f'{sum(history["time"])/60:.1f}분 경과', flush=True)

print('그래프 훅 준비 완료 — 기본 꺼짐. 켜려면 LIVE["on"] = True')

In [ ]:
# ══ 7. 학습 루프 ══════════════════════════════════════════════════════════
def run_epochs(model, opt, sam, crit, tl, vl, probe, epoch_range, cfg,
               hist, sharp, rho_fn, tag):
    se = cfg['total_epochs'] - cfg['decay_epochs']; last = epoch_range[-1]
    for epoch in epoch_range:
        lr_now = wsd_lr(epoch, cfg['total_epochs'], cfg['peak_lr'],
                        cfg['warmup_epochs'], cfg['decay_epochs'])
        for g in opt.param_groups: g['lr'] = lr_now
        rho_now = rho_fn(epoch) if rho_fn else 0.0
        sam.set_rho(rho_now)
        model.train()
        rl = rg = rn = 0.0; c = t = nb = 0; t0 = time.time()
        for x, y in tl:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            o = model(x); loss = crit(o, y); loss.backward()
            t += y.size(0); c += o.max(1)[1].eq(y).sum().item()
            rn += sam.grad_norm().item()
            gap = 0.0
            if sam.ascent():
                opt.zero_grad(set_to_none=True)
                la = crit(model(x), y); la.backward(); gap = la.item()-loss.item()
                sam.restore()
            if cfg['grad_clip']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step(); rl += loss.item(); rg += gap; nb += 1
        dt = time.time()-t0
        te = evaluate(model, vl)
        phase = ('warmup' if epoch < cfg['warmup_epochs']
                 else 'stable' if epoch < se else 'decay')
        for k, v in [('epoch',epoch+1),('phase',phase),('lr',lr_now),('rho',rho_now),
                     ('train_loss',rl/nb),('train_acc',100.0*c/t),('test_acc',te),
                     ('grad_norm',rn/nb),('ascent_gap',rg/nb),('time',dt)]:
            hist[k].append(v)
        extra = ''
        if (epoch+1) % cfg['sharpness_interval'] == 0 or epoch == last:
            o_, s_, g_ = measure_sharpness(model, crit, probe, cfg['sharpness_rho'])
            sharp['epoch'].append(epoch+1); sharp['one_sided'].append(o_)
            sharp['symmetric'].append(s_); sharp['probe_grad_norm'].append(g_)
            extra = f' | Sharp1 {o_:.4f} | SharpSym {s_:.6f} | pg {g_:.4f}'
        print(f'[{tag}|{phase:6s}] Ep [{epoch+1}/{cfg["total_epochs"]}] | {dt:.1f}s | '
              f'Loss {rl/nb:.4f} | Train {100.0*c/t:.2f}% | Eval {te:.2f}% | '
              f'lr {lr_now:.2e} | rho {rho_now:.4f} | gn {rn/nb:.3f}{extra}', flush=True)
        try: live_plot(tag, hist, sharp)
        except Exception as e: print('  (그래프 오류 무시:', e, ')', flush=True)

print('학습 루프 준비 완료')

In [ ]:
# ══ 8. Stage P / 1 / 2 ════════════════════════════════════════════════════
def stage_pretrain(cfg, pretrain_seed=42, force=False):
    """Tiny-ImageNet 사전학습. 모든 시드가 이 backbone 하나를 공유한다.
    연구 대상은 decay 구간이지 사전학습 분산이 아니므로 시드마다 다시 하지 않는다."""
    path = os.path.join(OUT_DIR, 'vit_pretrain_tinyimagenet.pt')
    if os.path.exists(path) and not force:
        print(f'Stage P 캐시 재사용: {path}', flush=True); return path
    root = prepare_tin()
    tr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(TIN_MEAN, TIN_STD, True, True))
    pr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(TIN_MEAN, TIN_STD, False, False))
    va = TinyImageNetVal(root, tr.class_to_idx,
                         transform=make_transforms(TIN_MEAN, TIN_STD, False, False))
    seed_everything(pretrain_seed)
    tl = DataLoader(tr, batch_size=cfg['batch_size'], shuffle=True, num_workers=2,
                    pin_memory=True, drop_last=True,
                    generator=loader_generator(pretrain_seed), worker_init_fn=worker_init)
    vl = DataLoader(va, batch_size=cfg['batch_size'], shuffle=False,
                    num_workers=2, pin_memory=True)
    pcfg = dict(total_epochs=cfg['pre_epochs'], warmup_epochs=3, decay_epochs=5,
                peak_lr=cfg['pre_lr'], grad_clip=1.0,
                sharpness_interval=5, sharpness_rho=cfg['sharpness_rho'])
    model = build_vit(len(tr.classes))
    opt = Lion(model.parameters(), lr=cfg['pre_lr'], weight_decay=cfg['weight_decay'])
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)   # ViT 엔 label smoothing 이 효과적
    hist, sharp = new_history()
    print(f'\n=== Stage P — Tiny-ImageNet ({cfg["pre_epochs"]} epochs) ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(cfg['pre_epochs']), pcfg, hist, sharp, None, 'P')
    torch.save({'model': model.state_dict(), 'cfg': pcfg,
                'num_classes': len(tr.classes), 'seed': pretrain_seed,
                'history': hist, 'sharpness_history': sharp}, path)
    print(f'\nStage P 저장: {path}', flush=True)
    print(f'  val acc {hist["test_acc"][-1]:.2f}% (200클래스 랜덤 0.5%) | '
          f'{sum(hist["time"])/60:.1f}분', flush=True)
    return path


def stage1(seed, cfg, pre_path, force=False):
    path = os.path.join(OUT_DIR, f'vit_stage1_seed{seed}.pt')
    if os.path.exists(path) and not force:
        print(f'Stage 1 캐시 재사용: {path}', flush=True); return path
    seed_everything(seed)
    tl, vl, pr = cifar_loaders(seed, cfg['batch_size'])
    model = build_vit(10)
    print('사전학습 backbone 이식 중...', flush=True)
    load_backbone(model, torch.load(pre_path, map_location=device,
                                    weights_only=False)['model'])
    opt = Lion(model.parameters(), lr=cfg['ft_lr'], weight_decay=cfg['weight_decay'])
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist, sharp = new_history()
    print(f'\n=== Stage 1 (seed {seed}) — CIFAR-10 {cfg["stable_end"]} epochs ===\n',
          flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(cfg['stable_end']), cfg, hist, sharp, None, f'S1-s{seed}')
    torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(),
                'cfg': cfg, 'seed': seed, 'history': hist,
                'sharpness_history': sharp}, path)
    print(f'\nStage 1 저장: {path}', flush=True)
    return path


def stage2(seed, rho, cfg, ckpt_path):
    tag = f'sam{rho:g}' if rho > 0 else 'samoff'
    arm = f'vit_seed{seed}_{tag}_d{cfg["decay_epochs"]}'
    out = os.path.join(OUT_DIR, f'arm_{arm}.json')
    if os.path.exists(out):
        print(f'skip {arm} (이미 있음)', flush=True)
        return json.load(open(out, encoding='utf-8'))
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    se = ck['cfg']['stable_end']

    # ★ 노트북판에 없던 부분 — 모든 arm 이 같은 데이터 순서를 보게 한다
    seed_everything(seed + 10_000)
    tl, vl, pr = cifar_loaders(seed + 10_000, cfg['batch_size'])

    model = build_vit(10); model.load_state_dict(ck['model'])
    opt = Lion(model.parameters(), lr=cfg['ft_lr'], weight_decay=cfg['weight_decay'])
    opt.load_state_dict(ck['optimizer'])            # 모멘텀까지 복원
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {k: list(v) for k, v in ck['history'].items()}
    sharp = {k: list(v) for k, v in ck['sharpness_history'].items()}

    def rho_fn(ep):
        if rho <= 0: return 0.0
        return rho * min(1.0, max(0.0, (ep-se+1)/max(1, cfg['rho_ramp_epochs'])))

    a0, o0, s0 = hist['test_acc'][-1], sharp['one_sided'][-1], sharp['symmetric'][-1]
    print(f'\n=== Stage 2 [{arm}] — 출발점 ep{se}: Test {a0:.2f}% | '
          f'SharpSym {s0:.6f} ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(se, cfg['total_epochs']), cfg, hist, sharp, rho_fn, f'S2-{arm}')

    best = max(hist['test_acc'])
    rec = {'arm': arm, 'seed': seed, 'arch': 'vit_tiny', 'rho_max': rho,
           'decay_epochs': cfg['decay_epochs'], 'stable_end': se, 'cfg': cfg,
           'results': {'decay_start_test_acc': a0, 'decay_start_one_sided': o0,
                       'decay_start_symmetric': s0,
                       'final_train_acc': hist['train_acc'][-1],
                       'final_test_acc': hist['test_acc'][-1], 'best_test_acc': best,
                       'final_gap': hist['train_acc'][-1]-hist['test_acc'][-1],
                       'final_one_sided': sharp['one_sided'][-1],
                       'final_symmetric': sharp['symmetric'][-1],
                       'final_probe_grad_norm': sharp['probe_grad_norm'][-1],
                       'decay_time_min': sum(hist['time'][se:])/60},
           'history': hist, 'sharpness_history': sharp,
           'timestamp': datetime.now().isoformat(timespec='seconds')}
    json.dump(rec, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    r = rec['results']
    print(f'\n=== [{arm}] 완료 ===', flush=True)
    print(f'  Test     {a0:.2f}% -> {r["final_test_acc"]:.2f}% '
          f'({r["final_test_acc"]-a0:+.2f}%p)', flush=True)
    print(f'  SharpSym {s0:.6f} -> {r["final_symmetric"]:.6f}', flush=True)
    print(f'  {r["decay_time_min"]:.1f}분 | 저장: {out}', flush=True)
    return rec

print('Stage 함수 준비 완료')

In [ ]:
# ══ 9. 설정 ═══════════════════════════════════════════════════════════════
SEEDS = [42, 43, 44]
RHOS  = [0.0, 0.05]

CFG = dict(stable_end=20, decay_epochs=10, warmup_epochs=2, total_epochs=30,
           pre_epochs=30, pre_lr=3e-4, ft_lr=1e-4, weight_decay=0.5,
           grad_clip=1.0, batch_size=256, rho_ramp_epochs=3,
           sharpness_interval=2, sharpness_rho=0.05)

print(f'Stage P 1회 + arm {len(SEEDS)*len(RHOS)}개 | 예상 약 2.1시간')

In [ ]:
# ══ 10. 실행 ══════════════════════════════════════════════════════════════
pre = stage_pretrain(CFG)                 # 36분, 한 번만
for seed in SEEDS:
    ck = stage1(seed, CFG, pre)
    for rho in RHOS:
        stage2(seed, rho, CFG, ck)
print('\n완료')

In [ ]:
# ══ 11. 결과 표 ═══════════════════════════════════════════════════════════
arms = [json.load(open(p, encoding='utf-8'))
        for p in sorted(glob.glob(os.path.join(OUT_DIR, 'arm_vit_*.json')))]
print(f'{"seed":>5}{"rho":>7}{"acc0":>8}{"acc1":>8}{"one":>9}{"sym":>12}{"|g|":>8}')
print('-'*57)
for a in sorted(arms, key=lambda x: (x['seed'], x['rho_max'])):
    r = a['results']
    print(f'{a["seed"]:>5}{a["rho_max"]:>7.2f}{r["decay_start_test_acc"]:>8.2f}'
          f'{r["final_test_acc"]:>8.2f}{r["final_one_sided"]:>9.4f}'
          f'{r["final_symmetric"]:>12.6f}{r["final_probe_grad_norm"]:>8.3f}')
print('-'*57)

# 시드별 paired 비교 (ResNet 의 Gate B 와 같은 형태)
from collections import defaultdict
by = defaultdict(dict)
for a in arms: by[a['seed']][a['rho_max']] = a['results']
pairs = [(s, d[0.0]['final_symmetric'], d[0.05]['final_symmetric'],
          d[0.0]['final_test_acc'], d[0.05]['final_test_acc'])
         for s, d in sorted(by.items()) if 0.0 in d and 0.05 in d]
if pairs:
    k = sum(1 for _,so,sn,_,_ in pairs if sn < so)
    print(f'\nsam_on 의 SharpSym 이 더 낮은 시드: {k}/{len(pairs)}')
    for s,so,sn,ao,an in pairs:
        print(f'  seed {s}: SharpSym {so:.6f} -> {sn:.6f} ({sn-so:+.6f}) | '
              f'acc {ao:.2f} -> {an:.2f} ({an-ao:+.2f}%p)')

In [ ]:
# ══ ★ 회수 (세션 끄기 전 반드시 실행) ═════════════════════════════════════
# Colab VM 은 임시다. Kaggle 에서 이미 한 번 전부 날렸다 (PROGRESS.md §8).
arc = os.path.join(OUT_DIR, 'phase2_vit_results.tar.gz')
with tarfile.open(arc, 'w:gz') as t:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.json'))
                    + glob.glob(os.path.join(OUT_DIR, '*.png'))):
        t.add(f, arcname=os.path.basename(f))
print(f'{arc}  ({os.path.getsize(arc)/1024:.0f} KB)')
try:
    from google.colab import files; files.download(arc)
except Exception as e:
    print('브라우저 다운로드 불가:', e)
    print('CLI 로:  colab download -s <세션>', arc, './results/')

# ★ 사전학습 backbone 은 용량이 커서 위 아카이브에 안 들어간다. 따로 받을 것.
pre_ck = os.path.join(OUT_DIR, 'vit_pretrain_tinyimagenet.pt')
print(f'\n사전학습 체크포인트: {pre_ck} '
      f'({os.path.getsize(pre_ck)/1024**2:.0f} MB)' if os.path.exists(pre_ck) else '')
try:
    from google.colab import files; files.download(pre_ck)
except Exception as e:
    print('CLI 로:  colab download -s <세션>', pre_ck, './results/')

---

# ==============================================================================
# 섹션 4 — Phase 0 — 기존 로그 재분석
# ==============================================================================

GPU 불필요. **이 섹션은 cwd 를 리포 루트로 바꾸므로 맨 뒤에 둔다.**

원본: `0_analysis.ipynb`  ·  셀 6개

# Phase 0 — 기존 로그 재분석 (새 학습 0분)

Kaggle 원본 산출물은 소실됐지만(`docs/PROGRESS.md` §8), 아카이브 노트북의 **출력 로그**에서
25개 run 의 에폭별 지표를 복구했다. 그것만으로 답할 수 있는 두 가지를 본다.

1. **arm 내부 궤적** — decay 구간에서 sharpness 와 accuracy 가 같이 움직이는가.
   raw 상관은 둘 다 시간에 단조라 생기는 허위 상관이므로 **1차 차분**으로 판단한다.
2. **across-run 산점도** — 측정 프로토콜이 3세대로 갈려 비교 가능한 Gen 2 만 쓴다.
   베이스라인 3종은 sharpness 측정 자체가 없다.

**GPU 불필요.** 로컬에서도 돈다.

In [ ]:
# ══ 1. 준비 ═══════════════════════════════════════════════════════════════
import os, csv, json, glob, math

# 리포 루트로 이동 (results/parsed 를 읽는다)
while not os.path.isdir('results/parsed') and os.getcwd() != '/':
    os.chdir('..')
print('작업 경로:', os.getcwd())

PARSED, OUTDIR = 'results/parsed', 'results/phase0'
os.makedirs(OUTDIR, exist_ok=True)

def mean(v): return sum(v)/len(v) if v else float('nan')
def pearson(x, y):
    n = len(x)
    if n < 3: return float('nan')
    mx, my = mean(x), mean(y)
    sx = sum((a-mx)**2 for a in x)**.5; sy = sum((b-my)**2 for b in y)**.5
    if sx == 0 or sy == 0: return float('nan')
    return sum((a-mx)*(b-my) for a,b in zip(x,y))/(sx*sy)
def ranks(v):
    r = [0]*len(v)
    for pos, i in enumerate(sorted(range(len(v)), key=lambda i: v[i])): r[i] = pos+1
    return r
def spearman(x, y): return pearson(ranks(x), ranks(y))
def diff(v): return [v[i+1]-v[i] for i in range(len(v)-1)]

def read_csv(p):
    with open(p) as f: return list(csv.DictReader(f))
def load(fn):
    p = os.path.join(PARSED, fn)
    return read_csv(p) if os.path.exists(p) else None

print('parsed run:', len(glob.glob(os.path.join(PARSED, '*.csv'))) - 1)

In [ ]:
# ══ 2. (필요시) 노트북 로그 재파싱 ════════════════════════════════════════
# 아카이브 노트북을 수정했거나 results/parsed 가 비었을 때만 실행하면 된다.
!python3 scripts/parse_notebook_logs.py

In [ ]:
# ══ 3. arm 내부 궤적 — 핵심 분석 ══════════════════════════════════════════
# decay 구간 안에서 sharpness 와 test acc 가 같이 움직이는가?
#
# 주의: raw 상관을 그대로 믿으면 안 된다. sam_on 에서는 sharpness 와 accuracy 가
# 둘 다 에폭에 대해 단조라, 상관이 −0.9 를 넘어도 그건 "둘 다 시간에 단조"라는
# 사실의 반영일 뿐이다. 1차 차분으로 추세를 제거한 Δ 로 판단한다.
DECAY_ARMS = [
    ('ResNet v1 decay10 sam_off', '09_wsd_sam_overlay_v1__c8.csv'),
    ('ResNet v1 decay10 sam_on',  '09_wsd_sam_overlay_v1__c9.csv'),
    ('ResNet v2 decay20 sam_off', '10_wsd_sam_overlay_v2__c8.csv'),
    ('ResNet v2 decay20 sam_on',  '10_wsd_sam_overlay_v2__c9.csv'),
    ('ViT decay10 sam_off',       '11_tiny_imagenet_wsd_sam__c7.csv'),
    ('ViT decay10 sam_on',        '11_tiny_imagenet_wsd_sam__c8.csv'),
]

traj = []
print(f'{"arm":<28}{"n":>3}{"raw":>8}{"rank":>8}{"e~s":>8}{"e~a":>8}{"Δ":>9}')
print('-'*72)
for name, fn in DECAY_ARMS:
    rows = load(fn)
    if not rows: print(f'{name:<28}  (파일 없음)'); continue
    rows = [r for r in rows if r['sharpness'] and r['test_acc']]
    if len(rows) < 3: print(f'{name:<28}  (표본 부족)'); continue
    e = [float(r['epoch']) for r in rows]
    s = [float(r['sharpness']) for r in rows]
    a = [float(r['test_acc']) for r in rows]
    rec = dict(arm=name, n=len(s), raw=pearson(s,a), rank=spearman(s,a),
               epoch_sharp=pearson(e,s), epoch_acc=pearson(e,a),
               detrended=pearson(diff(s), diff(a)),
               acc_start=a[0], acc_end=a[-1], sharp_start=s[0], sharp_end=s[-1])
    traj.append(rec)
    print(f'{name:<28}{rec["n"]:>3}{rec["raw"]:>8.3f}{rec["rank"]:>8.3f}'
          f'{rec["epoch_sharp"]:>8.3f}{rec["epoch_acc"]:>8.3f}{rec["detrended"]:>9.3f}')
print('-'*72)
print()
print('해석: e~a 가 +0.9 이상이면 accuracy 는 에폭에 대해 사실상 단조 상승이다.')
print('      sam_on 은 e~s 도 −0.9 근처라 raw 상관이 커 보이는 것이고,')
print('      Δ 의 부호가 arm 마다 뒤집히면 "추세를 빼면 무관"이 결론이다.')

In [ ]:
# ══ 4. across-run 산점도 (보조) ═══════════════════════════════════════════
# sharpness 측정 프로토콜이 3세대로 갈려 있어 전부를 한 축에 올릴 수 없다.
GEN = {'01': 'none', '02': 'gen1', '03': 'gen1', '04': 'gen0', '05': 'gen0',
       '06': 'gen0', '07': 'gen1', '08': 'gen2', '09': 'gen2', '10': 'gen2',
       '11': 'vit'}
LABEL = {'gen2': '✅ Gen 2 (ResNet, 고정 probe) — 비교 가능',
         'vit':  '△  Gen 2 (ViT) — loss·해상도가 달라 별도 축',
         'gen1': '❌ Gen 1 — probe 가 run 마다 다름',
         'gen0': '❌ Gen 0 — 매번 새 배치',
         'none': '—  sharpness 측정 없음'}

groups = {k: [] for k in LABEL}
for row in read_csv(os.path.join(PARSED, '_index.csv')):
    g = GEN.get(row['file'][:2], 'none')
    acc, sh = row.get('final_test_acc') or '', row.get('final_sharpness') or ''
    groups[g].append(dict(file=row['file'],
                          acc=float(acc) if acc else None,
                          sharp=float(sh) if sh else None))

usable = []
for g in ['gen2', 'vit', 'gen1', 'gen0', 'none']:
    if not groups[g]: continue
    print(f'{LABEL[g]}  ({len(groups[g])}개)')
    for it in groups[g]:
        a = f'{it["acc"]:.2f}%' if it['acc'] is not None else '   —  '
        s = f'{it["sharp"]:.4f}' if it['sharp'] is not None else '  —   '
        print(f'    acc {a:>8}  sharp {s:>8}  {it["file"]}')
        if g == 'gen2' and it['acc'] is not None and it['sharp'] is not None:
            usable.append(it)
    print()

# 같은 Stage 1 run 이 여러 노트북에 실려 중복 계수된다
seen, dedup = set(), []
for u in usable:
    k = (round(u['acc'],2), round(u['sharp'],4))
    if k in seen: print(f'  (중복 제외) {u["file"]}'); continue
    seen.add(k); dedup.append(u)

if len(dedup) >= 3:
    print(f'\nGen 2 상관 (중복 제거 n={len(dedup)}): '
          f'Pearson {pearson([u["sharp"] for u in dedup],[u["acc"] for u in dedup]):+.3f}')
print()
print('⚠️ 이 상관을 증거로 쓰면 안 된다.')
print('   남은 점들도 서로 독립이 아니다 — 각 노트북의 두 arm 은 같은 Stage 1')
print('   체크포인트에서 분기해 출발점을 공유한다. 유효 표본은 독립 run 3개 수준이고,')
print('   신뢰구간이 사실상 (-1, 1) 전체다. 주장은 위 Δ(차분 상관)에 싣는다.')
print()
print('⚠️ 베이스라인 3종(SGD 94.22 / AdamW 92.93 / SAM 94.69)에는 sharpness 가 없다.')
print('   고정확도 구간(92~95%)이 통째로 비어 있다는 뜻이다.')

In [ ]:
# ══ 5. 그림 ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, fn in DECAY_ARMS:
    rows = load(fn)
    if not rows: continue
    rows = [r for r in rows if r['sharpness'] and r['test_acc']]
    e = [float(r['epoch']) for r in rows]
    on = 'sam_on' in name
    style = '-' if on else '--'
    color = ('tab:blue' if on else 'tab:red') if name.startswith('ViT') \
            else ('tab:green' if on else 'tab:orange')
    axes[0].plot(e, [float(r['test_acc']) for r in rows], style,
                 color=color, marker='o', ms=3, label=name)
    axes[1].plot(e, [float(r['sharpness']) for r in rows], style,
                 color=color, marker='o', ms=3, label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy rises during decay')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Sharpness (one-sided)')
axes[1].set_title('Sharpness does not follow')
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUTDIR}/fig1_decay_trajectories.png',
                                dpi=150, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
xs = range(len(traj))
ax.bar([x-.2 for x in xs], [t['raw'] for t in traj], .4,
       label='raw correlation', color='tab:gray')
ax.bar([x+.2 for x in xs], [t['detrended'] for t in traj], .4,
       label='detrended (first difference)', color='tab:purple')
ax.axhline(0, color='k', lw=.8)
ax.set_xticks(list(xs)); ax.set_xticklabels([t['arm'] for t in traj],
                                            rotation=20, ha='right', fontsize=8)
ax.set_ylabel('corr(sharpness, accuracy)')
ax.set_title('Raw correlation is a time-trend artifact')
ax.legend(fontsize=8); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'{OUTDIR}/fig2_detrended.png',
                                dpi=150, bbox_inches='tight'); plt.show()

json.dump({'trajectories': traj, 'gen2_scatter': dedup},
          open(f'{OUTDIR}/phase0_results.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
print('저장:', OUTDIR)